In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from wordcloud import WordCloud

# ------------------------
# 1. Read Data and Construct Graph
# ------------------------
inputFilename = 'updated_chatgpt_reddit_comments.csv'
commentsDataFrame = pd.read_csv(inputFilename)

# Ensure the IDs are treated as strings
commentsDataFrame['comment_id'] = commentsDataFrame['comment_id'].astype(str)
commentsDataFrame['comment_parent_id'] = commentsDataFrame['comment_parent_id'].astype(str)

# Extract columns
commentIds = commentsDataFrame['comment_id']
parentCommentIds = commentsDataFrame['comment_parent_id']

# Collect all unique nodes
uniqueNodeIds = pd.unique(pd.concat([commentIds, parentCommentIds]))

# Create a directed graph representing comment thread structure
commentThreadGraph = nx.DiGraph()
commentThreadGraph.add_nodes_from(uniqueNodeIds)
commentEdges = list(zip(commentIds, parentCommentIds))
commentThreadGraph.add_edges_from(commentEdges)
# Remove self loops
commentThreadGraph.remove_edges_from(nx.selfloop_edges(commentThreadGraph))

# ------------------------
# 2. Plot the Graph
# ------------------------
# Use a spring layout (force-directed)
nodePositions = nx.spring_layout(commentThreadGraph, seed=42)  # seed for reproducibility

graphFigure, graphAxes = plt.subplots(figsize=(12, 8))
nx.draw_networkx_nodes(commentThreadGraph, nodePositions, node_size=50, ax=graphAxes)
nx.draw_networkx_edges(commentThreadGraph, nodePositions, ax=graphAxes, arrows=False, alpha=0.3)
graphAxes.set_title('Comment Thread Graph')
graphAxes.axis('off')

# Global storage for use in callback
globalGraphData = {
    'commentThreadGraph': commentThreadGraph,
    'commentsDataFrame': commentsDataFrame,
    'nodePositions': nodePositions
}

# ------------------------
# 3. Define Mouse Click Callback Function
# ------------------------
def handleNodeClick(clickEvent):
    # Ignore clicks outside the axes
    if clickEvent.xdata is None or clickEvent.ydata is None:
        return
    clickPoint = np.array([clickEvent.xdata, clickEvent.ydata])
    
    # Find the nearest node by comparing distances from clickPoint to node positions
    nodePositionsArray = np.array(list(globalGraphData['nodePositions'].values()))
    nodeIdsList = list(globalGraphData['nodePositions'].keys())
    distancesToNodes = np.linalg.norm(nodePositionsArray - clickPoint, axis=1)
    minimumDistance = np.min(distancesToNodes)
    nearestNodeIndex = np.argmin(distancesToNodes)
    
    # Define a threshold (may need tuning depending on your layout scale)
    clickDistanceThreshold = 0.05
    if minimumDistance > clickDistanceThreshold:
        print("Click was not close enough to any node.")
        return
    
    clickedCommentId = nodeIdsList[nearestNodeIndex]
    print("Clicked comment_id:", clickedCommentId)
    
    # Retrieve corresponding serial_number from the data table (if available)
    matchingRow = globalGraphData['commentsDataFrame'][globalGraphData['commentsDataFrame']['comment_id'] == clickedCommentId]
    if not matchingRow.empty:
        serialNumber = matchingRow.iloc[0]['serial_number']
        print("Serial number for clicked comment:", serialNumber)
    else:
        print("Clicked comment_id not found in data table.")
    
    # ------------------------
    # Find Descendant Nodes
    # ------------------------
    # Reverse the graph to follow child-to-parent edges
    reversedGraph = globalGraphData['commentThreadGraph'].reverse(copy=False)
    # Use networkx.descendants to get all nodes reachable from clickedCommentId
    descendantCommentIds = nx.descendants(reversedGraph, clickedCommentId)
    print("Descendant nodes:", descendantCommentIds)
    
    # ------------------------
    # Filter Data and Display Comment Details
    # ------------------------
    allRelatedCommentIds = set([clickedCommentId]) | descendantCommentIds
    filteredCommentsDataFrame = globalGraphData['commentsDataFrame'][globalGraphData['commentsDataFrame']['comment_id'].isin(allRelatedCommentIds)]
    print(f"Performing topic analysis on {len(filteredCommentsDataFrame)} comments...")
    
    # Build a multi-line string with comment details using vectorized operations
    # This is more efficient than iterrows() which creates Series objects per row
    commentDetailsString = '\n\n'.join(
        f"comment_id: {commentId} | serial_number: {serialNum}\nComment: {commentBody}"
        for commentId, serialNum, commentBody in zip(filteredCommentsDataFrame['comment_id'], filteredCommentsDataFrame['serial_number'], filteredCommentsDataFrame['comment_body'])
    )
    
    # Display details in a new figure (simple text display)
    detailsFigure, detailsAxes = plt.subplots(figsize=(8, 6))
    detailsAxes.text(0.01, 0.99, commentDetailsString, va='top', ha='left', fontsize=8, wrap=True)
    detailsAxes.axis('off')
    detailsFigure.canvas.manager.set_window_title("Comment Details")
    plt.show(block=False)
    
    # ------------------------
    # Clean and Tokenize the Comment Text
    # ------------------------
    # Convert comment text to lowercase
    commentTextCorpus = filteredCommentsDataFrame['comment_body'].astype(str).str.lower().tolist()
    tfidfVectorizerModel = TfidfVectorizer(stop_words='english')
    tfidfMatrix = tfidfVectorizerModel.fit_transform(commentTextCorpus)
    
    # ------------------------
    # Latent Semantic Analysis (LSA) and Word Cloud
    # ------------------------
    # Use TruncatedSVD to perform SVD on the TF-IDF matrix
    numTopicComponents = 5  # Number of topics to extract (can be adjusted)
    svdModel = TruncatedSVD(n_components=numTopicComponents, random_state=42)
    svdModel.fit(tfidfMatrix)
    # Extract the first latent topic (first column of V)
    rightSingularVectors = svdModel.components_.T  # shape: (n_terms, n_components)
    topicWeightVector = rightSingularVectors[:, 0]
    vocabularyWords = tfidfVectorizerModel.get_feature_names_out()
    
    # Sort words by absolute weight in descending order
    sortedWordIndices = np.argsort(np.abs(topicWeightVector))[::-1]
    topWordsCount = min(20, len(vocabularyWords))
    topTopicWords = vocabularyWords[sortedWordIndices[:topWordsCount]]
    topTopicWeights = topicWeightVector[sortedWordIndices[:topWordsCount]]
    
    # Generate a word cloud (using absolute values of weights)
    wordFrequencyDict = {word: float(abs(weight)) for word, weight in zip(topTopicWords, topTopicWeights)}
    wordCloudGenerator = WordCloud(width=800, height=400, background_color='white')
    wordCloudGenerator.generate_from_frequencies(wordFrequencyDict)
    
    plt.figure()
    plt.imshow(wordCloudGenerator, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Latent Semantic Topic for comment_id: {clickedCommentId}")
    plt.show()
    
    # Print top words and their weights
    print("Top words in the extracted topic:")
    for word, weight in zip(topTopicWords, topTopicWeights):
        print(f"{word} (weight: {weight:.3f})")

# ------------------------
# 4. Attach the Click Callback to the Figure
# ------------------------
clickCallbackId = graphFigure.canvas.mpl_connect('button_press_event', handleNodeClick)
plt.show()
